# Capítulo 8 — ML para cibersegurança: detecção de intrusão em IIoT

## Missão
Agora o ML do lado da defesa. Os capítulos 13 a 15 do livro comparam algoritmos para **detecção de intrusão** em redes industriais (IIoT). Com 12 mil fluxos de rede e 4 tipos de ataque (DoS, varredura, força bruta e exfiltração), você vai:
- treinar detectores **supervisionados** (binário e multiclasse);
- medir a taxa de detecção por tipo de ataque e os falsos alarmes;
- montar um detector de **anomalias** sem rótulos (Isolation Forest);
- testar o pior cenário: um ataque **novo**, que não estava no treino.

**No fim deste caderno você terá:** o módulo `meu_escudo/ids.py` (preparação dos fluxos, relatório por tipo de ataque e detector de anomalias calibrado por falsos alarmes) e a comparação supervisionado × anomalia em ataques novos.

**Leitura essencial (opcional, livro em inglês):**
- 13.5 Different Machine Learning Techniques In Cyber Security · PDF p. 186
- 13.7 Deep Learning Techniques In Cyber Security · PDF p. 193
- 14.3 Research Methodology · PDF p. 234
- 15.2 IDS Applications · PDF p. 257
- 15.3 Use Of ML And DL Algorithms In IIOT Applications · PDF p. 260
- 15.4 Practical Application of ML Algorithms In IIOT · PDF p. 261

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap08_ml_para_ciberseguranca_deteccao_de_intrusao_e.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados fictícios do projeto)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_escudo/
os.makedirs("../meu_escudo", exist_ok=True)
open("../meu_escudo/__init__.py", "a").close()        # meu_escudo/ é o projeto que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
import warnings; warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: meu_escudo")

## Ideia em 30 segundos
Um IDS (sistema de detecção de intrusão) tem duas filosofias:
- **assinaturas/supervisionado**: aprende os ataques conhecidos, com ótima precisão, mas cego para o novo;
- **anomalia**: aprende o normal e alerta o que foge dele; pega o novo, com mais falsos alarmes.

Métricas que importam para o SOC (centro de operações de segurança):
- a **taxa de detecção** por tipo de ataque;
- os **falsos alarmes por dia**: cada um custa tempo de um analista.

### 8.1 **O módulo do IDS.** Crie `ids.py` com:
- `preparar(df)`: as numéricas + as dummies de `protocolo`, e `log1p` em `bytes_env`, `bytes_rec` e `pacotes`;
- `relatorio_por_tipo(tipos, pred_ataque)`: para cada tipo, a fração marcada como ataque (para `normal`, é a taxa de falso alarme);
- `calibrar_limiar(escores_normais, falsos=0.01)`: o quantil que dá a fração desejada de falsos alarmes.

_A célula grava `meu_escudo/ids.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_escudo/ids.py


### 8.2 **Supervisionado binário.** Carregue `fluxos_rede()`, prepare, separe 70/30 estratificado pelo tipo e compare uma `LogisticRegression` padronizada, uma `RandomForestClassifier(200)` e um `MLPClassifier((64, 32))`: AUC e o relatório por tipo da melhor.

### 8.3 **Multiclasse.** Treine a Random Forest para prever o **tipo** de fluxo e mostre o `classification_report`. Qual ataque é mais confundido com o tráfego normal? Mostre as 6 características mais importantes.

### 8.4 **Detector de anomalias.** Treine um `IsolationForest(300)` **só com o tráfego normal** do treino. Use `-score_samples` como escore de anomalia e calibre o limiar para 1% de falsos alarmes nos normais do treino. Mostre o relatório por tipo no teste.

### 8.5 **O ataque novo (dia zero).** Retreine a Random Forest **sem nenhum exemplo de exfiltração** no treino (tire esse tipo do treino) e compare, no teste, a taxa de detecção da exfiltração pela RF e pelo Isolation Forest. Qual filosofia protege contra o desconhecido?

### 8.6 **Falsos alarmes por dia.** Suponha 2 milhões de fluxos normais por dia. Converta a taxa de falso alarme do RF e a do Isolation Forest (limiar de 1%) em alertas por dia. Quantos analistas, a 200 alertas por dia cada, cada abordagem exige? Combine as duas: alerta quando o RF **ou** a anomalia muito forte (limiar de 0,1%) disparam.

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
